# Phase 1 Measurement Workflow (Debug/Bring-up)

This notebook is intentionally short and operational:
- one setup block
- one preflight-only block
- one run+process+save block
- one plot block for last run
- one plot block for last N runs
- one time-domain debug block for selected frequency/repeat
- one frequency-domain debug block for selected frequency/repeat

This notebook works with real hardware only (no fake adapter).


In [ ]:
# BLOCK 1 - Setup (run once)
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "eis").exists() and (candidate / "USB6451").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory.")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from eis import (
    CaptureConditioningConfig,
    ExcitationConfig,
    HardwareConfig,
    ImpedanceProcessingConfig,
    RunSaveOptions,
    RunSelection,
    load_and_validate_config,
    plot_capture_fft_components,
    plot_capture_time_domain_components,
    plot_impedance_bode,
    plot_impedance_inverse_nyquist,
    plot_impedance_nyquist,
    plot_snr_vs_frequency,
    run_measure_process_save,
    run_preflight_only,
)

# ----------------------------- User Inputs -----------------------------
CONFIG_PATH = REPO_ROOT / "config_examples" / "config_phase1_example.xlsx"
BASE_OUTPUT_DIR = REPO_ROOT / "measurements"
SERIAL_NUMBER = f"PH1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
USER_NAME = "operator"
DESCRIPTION = ""
REPEATS = 1
RUN_PREFLIGHT_DURING_SWEEP = False

# ----------------------------- Save Options ----------------------------
save_options = RunSaveOptions(
    write_metadata_bank_txt=True,
    write_metadata_bank_csv=True,
    write_metadata_report_html=True,
    write_metadata_report_pdf=False,
    write_description_file=True,
)
SAVE_PLOTS_PNG = True
SAVE_PLOTS_VECTOR = True

# ------------------------ Hardware/Excitation --------------------------
hardware = HardwareConfig(
    device="Dev1",
    ao_channel="ao0",
    ai_channels=("ai0", "ai7"),
    input_mode="differential",
)

excitation = ExcitationConfig(
    drive_mode="auto_from_current_rms",
    offset_v=0.0,
    manual_current_range="20A",
)

conditioning = CaptureConditioningConfig(
    settle_discard_s=0.15,
    extra_periods_for_trim=1,
    alignment_search_periods=1,
)

processing = ImpedanceProcessingConfig(
    method="fft",
    sine_fit_backend="numpy_lstsq",
    filter_mode="lowpass",
    lowpass_cutoff_hz=2000.0,
    shunt_resistance_ohm=0.008,
)

# ----------------------------- Preflight -------------------------------
PREFLIGHT_SAMPLE_RATE_SPS = None
PREFLIGHT_SAMPLES_PER_CHANNEL = None
PREFLIGHT_TEST_CURRENT_RMS_A = 10.0
PREFLIGHT_MANUAL_CURRENT_RANGE = "20A"
PREFLIGHT_SHUNT_RESISTANCE_OHM = 0.008
PREFLIGHT_SHUNT_TOLERANCE_PERCENT = 15.0
PREFLIGHT_CURRENT_CHANNEL_INDEX = 0
PREFLIGHT_SETTLE_DISCARD_S = 0.15

# ------------------------- Plot Selection ------------------------------
LAST_N_FOR_OVERLAY = 3
SERIAL_FILTER_FOR_OVERLAY = SERIAL_NUMBER

# Time-domain debug selection (independent)
TD_REPEAT_INDEX = 1
TD_COMPONENTS = ("raw", "filtered", "fitted")
TD_CHANNEL_INDICES = (0, 1)

# Frequency-domain debug selection (independent)
FD_REPEAT_INDEX = 1
FD_COMPONENTS = ("raw", "filtered", "fitted")
FD_CHANNEL_INDICES = (0, 1)
FD_MAX_FREQUENCY_HZ = None

sweep = load_and_validate_config(CONFIG_PATH)
DEFAULT_DEBUG_FREQUENCY_HZ = float(sweep.points[0].frequency_hz)
TD_FREQUENCY_HZ = DEFAULT_DEBUG_FREQUENCY_HZ
FD_FREQUENCY_HZ = DEFAULT_DEBUG_FREQUENCY_HZ

print(f"Repo root: {REPO_ROOT}")
print(f"Config rows: {len(sweep.points)}")
print("Run mode: REAL HARDWARE")
print(f"Serial number: {SERIAL_NUMBER}")
print(f"Default debug frequency: {DEFAULT_DEBUG_FREQUENCY_HZ:g} Hz")


In [ ]:
# BLOCK 2 - Preflight Only (no measurement sweep)
preflight_only_result = run_preflight_only(
    sweep=sweep,
    hardware=hardware,
    excitation=excitation,
    sample_rate_sps=PREFLIGHT_SAMPLE_RATE_SPS,
    samples_per_channel=PREFLIGHT_SAMPLES_PER_CHANNEL,
    test_current_rms_a=PREFLIGHT_TEST_CURRENT_RMS_A,
    manual_current_range=PREFLIGHT_MANUAL_CURRENT_RANGE,
    shunt_resistance_ohm=PREFLIGHT_SHUNT_RESISTANCE_OHM,
    shunt_voltage_tolerance_percent=PREFLIGHT_SHUNT_TOLERANCE_PERCENT,
    current_channel_index=PREFLIGHT_CURRENT_CHANNEL_INDEX,
    settle_discard_s=PREFLIGHT_SETTLE_DISCARD_S,
)

print("Preflight-only completed")
print(f"  Sample rate: {preflight_only_result.sample_rate_sps:g} S/s")
print(f"  Samples/ch : {preflight_only_result.samples_per_channel}")
print(f"  Shape      : {preflight_only_result.measured_shape}")
print(f"  Message    : {preflight_only_result.message}")


In [ ]:
# BLOCK 3 - Run, process, save (+ validation summary)
run_bundle = run_measure_process_save(
    sweep=sweep,
    hardware=hardware,
    excitation=excitation,
    processing=processing,
    base_output_dir=BASE_OUTPUT_DIR,
    serial_number=SERIAL_NUMBER,
    user_name=USER_NAME,
    description=DESCRIPTION,
    repeats=REPEATS,
    run_preflight_during_sweep=RUN_PREFLIGHT_DURING_SWEEP,
    preflight_sample_rate_sps=PREFLIGHT_SAMPLE_RATE_SPS,
    preflight_samples_per_channel=PREFLIGHT_SAMPLES_PER_CHANNEL,
    preflight_test_current_rms_a=PREFLIGHT_TEST_CURRENT_RMS_A,
    preflight_manual_current_range=PREFLIGHT_MANUAL_CURRENT_RANGE,
    preflight_shunt_resistance_ohm=PREFLIGHT_SHUNT_RESISTANCE_OHM,
    preflight_shunt_voltage_tolerance_percent=PREFLIGHT_SHUNT_TOLERANCE_PERCENT,
    preflight_current_channel_index=PREFLIGHT_CURRENT_CHANNEL_INDEX,
    preflight_settle_discard_s=PREFLIGHT_SETTLE_DISCARD_S,
    conditioning=conditioning,
    save_options=save_options,
)

LAST_RUN_BUNDLE = run_bundle
LAST_RUN_ROOT = run_bundle.layout.root
LAST_RUN_PLOTS_DIR = run_bundle.layout.plots
LAST_RUN_SERIAL = SERIAL_NUMBER

available_frequency_repeat = sorted(
    {(float(c.frequency_hz), int(c.repeat_index)) for c in run_bundle.run_result.captures}
)

print("Run completed")
print(f"  Run folder      : {LAST_RUN_ROOT}")
print(f"  Captures        : {len(run_bundle.run_result.captures)}")
print(f"  Impedance rows  : {len(run_bundle.impedance_results)}")
print(f"  Raw artifacts   : {len(run_bundle.persisted_artifacts.capture_artifacts)}")
print(f"  Point summaries : {len(run_bundle.persisted_artifacts.point_summaries)}")
if run_bundle.run_result.preflight is None:
    print("  Sweep preflight : skipped")
else:
    print(f"  Sweep preflight : {run_bundle.run_result.preflight.message}")
print("  Saved files:")
for item in run_bundle.saved_paths:
    print(f"    - {item}")
print("  Available (frequency_hz, repeat_index):")
for frequency_hz, repeat_index in available_frequency_repeat:
    print(f"    - ({frequency_hz:.6g}, {repeat_index})")


In [ ]:
# BLOCK 4 - Plots for last run
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so LAST_RUN_BUNDLE is available.")

selection_last = RunSelection(mode="last", serial_numbers=(LAST_RUN_SERIAL,))

last_run_plot_paths = []

def _last_png(name: str):
    if not SAVE_PLOTS_PNG:
        return None
    return LAST_RUN_PLOTS_DIR / f"{name}.png"

nyquist_png = _last_png("last_run_nyquist")
inv_nyquist_png = _last_png("last_run_inverse_nyquist")
bode_png = _last_png("last_run_bode")
snr_png = _last_png("last_run_snr")

plot_impedance_nyquist(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    save_path=nyquist_png,
)
plot_impedance_inverse_nyquist(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    save_path=inv_nyquist_png,
)
plot_impedance_bode(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    save_path=bode_png,
)
plot_snr_vs_frequency(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    snr_source="current",
    threshold_db=20.0,
    good_region="above_threshold",
    save_path=snr_png,
)

for item in (nyquist_png, inv_nyquist_png, bode_png, snr_png):
    if item is not None:
        last_run_plot_paths.append(item)

print("Last-run plots generated")
for item in last_run_plot_paths:
    print(f"  - {item}")


In [ ]:
# BLOCK 5 - Plots for last N runs (set N here)
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so output paths are available.")

N = int(LAST_N_FOR_OVERLAY)
if N < 1:
    raise ValueError("LAST_N_FOR_OVERLAY must be >= 1")

selection_last_n = RunSelection(
    mode="last_n",
    last_n=N,
    serial_contains=SERIAL_FILTER_FOR_OVERLAY,
)

nyquist_last_n = LAST_RUN_PLOTS_DIR / f"last_{N}_nyquist.png"
bode_last_n = LAST_RUN_PLOTS_DIR / f"last_{N}_bode.png"
snr_last_n = LAST_RUN_PLOTS_DIR / f"last_{N}_snr.png"

_, _, runs_ny = plot_impedance_nyquist(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last_n,
    save_path=(nyquist_last_n if SAVE_PLOTS_PNG else None),
)
_, _, runs_bd = plot_impedance_bode(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last_n,
    save_path=(bode_last_n if SAVE_PLOTS_PNG else None),
)
_, _, runs_sr, _ = plot_snr_vs_frequency(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last_n,
    snr_source="current",
    threshold_db=20.0,
    good_region="above_threshold",
    save_path=(snr_last_n if SAVE_PLOTS_PNG else None),
)

print(f"Last-{N} overlay plots generated")
print("  Runs used (Nyquist):")
for item in runs_ny:
    print(f"    - {item.root.name}")
print("  Runs used (Bode):")
for item in runs_bd:
    print(f"    - {item.root.name}")
print("  Runs used (SNR):")
for item in runs_sr:
    print(f"    - {item.root.name}")
if SAVE_PLOTS_PNG:
    print(f"  - {nyquist_last_n}")
    print(f"  - {bode_last_n}")
    print(f"  - {snr_last_n}")


In [ ]:
# BLOCK 6 - Time-domain debug plot for selected frequency/repeat/components
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so LAST_RUN_BUNDLE is available.")

td_png = (
    LAST_RUN_PLOTS_DIR
    / f"debug_time_f{TD_FREQUENCY_HZ:.6g}_rep{TD_REPEAT_INDEX}.png"
    if SAVE_PLOTS_PNG
    else None
)
td_svg = (
    LAST_RUN_PLOTS_DIR / f"debug_time_f{TD_FREQUENCY_HZ:.6g}_rep{TD_REPEAT_INDEX}.svg"
    if SAVE_PLOTS_VECTOR
    else None
)

_, _, td_result = plot_capture_time_domain_components(
    run_result=LAST_RUN_BUNDLE.run_result,
    frequency_hz=TD_FREQUENCY_HZ,
    repeat_index=TD_REPEAT_INDEX,
    components=TD_COMPONENTS,
    processing_config=processing,
    channel_indices=TD_CHANNEL_INDICES,
    print_snr_table=True,
    save_path=td_png,
    save_vector_path=td_svg,
)

print("Time-domain debug plot generated")
print(
    f"  Selected: row={td_result.row_number}, repeat={td_result.repeat_index}, "
    f"f={td_result.frequency_hz:.6g} Hz"
)
if td_png is not None:
    print(f"  - {td_png}")
if td_svg is not None:
    print(f"  - {td_svg}")


In [ ]:
# BLOCK 7 - Frequency-domain debug plot for selected frequency/repeat/components
if "LAST_RUN_BUNDLE" not in globals():
    raise RuntimeError("Run BLOCK 3 first so LAST_RUN_BUNDLE is available.")

fd_png = (
    LAST_RUN_PLOTS_DIR
    / f"debug_fft_f{FD_FREQUENCY_HZ:.6g}_rep{FD_REPEAT_INDEX}.png"
    if SAVE_PLOTS_PNG
    else None
)
fd_svg = (
    LAST_RUN_PLOTS_DIR / f"debug_fft_f{FD_FREQUENCY_HZ:.6g}_rep{FD_REPEAT_INDEX}.svg"
    if SAVE_PLOTS_VECTOR
    else None
)

_, _, fd_result = plot_capture_fft_components(
    run_result=LAST_RUN_BUNDLE.run_result,
    frequency_hz=FD_FREQUENCY_HZ,
    repeat_index=FD_REPEAT_INDEX,
    components=FD_COMPONENTS,
    processing_config=processing,
    channel_indices=FD_CHANNEL_INDICES,
    max_frequency_hz=FD_MAX_FREQUENCY_HZ,
    print_snr_table=True,
    save_path=fd_png,
    save_vector_path=fd_svg,
)

print("Frequency-domain debug plot generated")
print(
    f"  Selected: row={fd_result.row_number}, repeat={fd_result.repeat_index}, "
    f"f={fd_result.frequency_hz:.6g} Hz"
)
if fd_png is not None:
    print(f"  - {fd_png}")
if fd_svg is not None:
    print(f"  - {fd_svg}")
